In [24]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import statsmodels.formula.api as smf
from scipy.optimize import fmin_slsqp
from toolz import reduce, partial
import pyfixest as pf
from pyfixest import iplot

In [2]:
aqi_data = pd.read_csv("aqi_daily_1980_to_2021.csv")

In [14]:
rb_states = [
    "Illinois",
    "Indiana",
    "Michigan",
    "New York",
    "Ohio",
    "Pennsylvania",
    "West Virginia",
    "Wisconsin"
]

rb_aqi_data = aqi_data[aqi_data["State Name"].isin(rb_states)]

rb_aqi_data["Date"] = pd.to_datetime(rb_aqi_data["Date"])
rb_aqi_data["year"] = rb_aqi_data["Date"].dt.year

rb_aqi_summary = (rb_aqi_data.groupby(["County Name", "Defining Parameter", "year"], as_index=False)["AQI"]
                  .mean().rename(columns={"AQI": "avg_aqi"}))

rb_aqi_summary = rb_aqi_summary[rb_aqi_summary["year"] >= 2000]
rb_aqi_summary
#rb_aqi_summary.to_csv("aqi_summary.csv", index=False)

/var/folders/g0/1_gj42z94q93wf4k_ck_hqd40000gn/T/ipykernel_29932/2767879179.py:14: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  rb_aqi_data["Date"] = pd.to_datetime(rb_aqi_data["Date"])
/var/folders/g0/1_gj42z94q93wf4k_ck_hqd40000gn/T/ipykernel_29932/2767879179.py:15: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  rb_aqi_data["year"] = rb_aqi_data["Date"].dt.year


,County Name,Defining Parameter,year,avg_aqi
1,Adams,CO,2011,1.0
2,Adams,CO,2012,7.0
3,Adams,CO,2013,0.0
4,Adams,CO,2014,0.5
9,Adams,NO2,2002,25.0
...,...,...,...,...
18335,York,SO2,2013,27.0
18336,York,SO2,2014,46.0
18337,York,SO2,2015,56.0
18338,York,SO2,2017,51.0


In [21]:
rb_aqi_summary["treated"] = (rb_aqi_summary["County Name"] == "Wayne").astype(int)

rb_aqi_wide = (
    rb_aqi_summary
      .pivot_table(
          index=["County Name", "year"],     
          columns="Defining Parameter",
          values="avg_aqi",
          aggfunc="mean"
      )
      .reset_index()
)
rb_aqi_wide.columns.name = None

rb_aqi_wide["treated"] = (rb_aqi_wide["County Name"] == "Wayne").astype(int)

rb_aqi_wide.dropna(inplace=True)



In [23]:
rb_aqi_wide

,County Name,year,CO,NO2,Ozone,PM10,PM2.5,SO2,treated
484,Bucks,2006,24.000000,31.260504,62.243421,16.500000,49.983871,31.315789,0
546,Cambria,2002,28.000000,26.704545,66.757353,35.615385,58.618421,46.789474,0
547,Cambria,2003,14.500000,29.666667,51.118644,34.777778,59.975000,52.029412,0
548,Cambria,2004,30.000000,28.637931,49.808333,34.133333,57.512195,55.755556,0
552,Cambria,2008,18.000000,28.844828,46.075630,35.000000,54.388889,45.965909,0
...,...,...,...,...,...,...,...,...,...
3371,Saint Clair,2006,24.000000,28.945055,53.955975,37.500000,54.592593,41.076923,0
3918,Vanderburgh,2000,21.875000,28.897059,63.635659,59.000000,61.147727,56.830986,0
3919,Vanderburgh,2001,23.230769,25.964912,54.862319,47.500000,59.750000,46.148148,0
3927,Vanderburgh,2009,8.875000,17.172414,46.214953,30.000000,51.665072,5.454545,0
